# Prepare Retrain Data

Selects crops from inference results and copies them to a review folder
for human labeling via `crop_labeler_v2.py`.

**This is a deliberate, manual step.** Nothing is moved automatically.
You choose which crops to review, review them, then manually merge into `labeled_crops/`.

**Typical use cases:**
- After running inference: send low-confidence crops for review
- Hard negative mining: send background crops that were difficult for the model
- Disagreement review: send crops where two pipelines disagree

**Workflow:**
```
infer_cropbased results
       ↓  (this notebook)
retrain_review/{class}/   ← crop_labeler_v2.py reviews here
       ↓  (manual merge after review)
labeled_crops/{class}/    ← retrain
```

## Cell 1 — Environment

Set your local path. Only edit `BASE_DIR`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/pollinator-classification')
else:
    BASE_DIR = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')

IMAGE_ROOT        = BASE_DIR / 'Insects_images' / 'e2e_evaluation_images'
GT_ANN_ROOT       = BASE_DIR / 'Insects_images' / 'e2e_yolo_annotations'
MODEL_DIR         = BASE_DIR / 'models'
CROP_RESULTS_ROOT = BASE_DIR / 'Insects_images' / 'crop_results'
YOLO_RESULTS_ROOT = BASE_DIR / 'Insects_images' / 'yolo_results'
INSECTNET_W       = BASE_DIR / 'InsectNet' / 'model.pth'
LABELED_DIR       = BASE_DIR / 'Insects_images' / 'annotated_crops'

CROP_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR         : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'IMAGE_ROOT       : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'GT_ANN_ROOT      : {GT_ANN_ROOT}  exists={GT_ANN_ROOT.exists()}')
print(f'MODEL_DIR        : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'CROP_RESULTS_ROOT: {CROP_RESULTS_ROOT}')
print(f'YOLO_RESULTS_ROOT: {YOLO_RESULTS_ROOT}')


## Cell 2 — Filter config  ← **edit this**

Controls which crops are selected for review:

- **`INFER_RESULTS`** — which inference run to draw from
- **`CONF_THRESHOLD_LOW`** — crops with confidence below this are flagged as uncertain
  and sent for review. Lower = more crops to review.
- **`INCLUDE_BACKGROUND`** — include background crops (useful as hard negatives for retraining)
- **`FORCE_ALL`** — send everything regardless of confidence (use sparingly)
- **`MAX_PER_CLASS`** — cap per class to keep the review session manageable

In [ ]:
# ── Which pipeline results to use ───────────────────────────────
# Point to the results folder from infer_cropbased
# e.g. RESULTS_ROOT / 'results_five_class' or just RESULTS_ROOT
INFER_RESULTS = RESULTS_ROOT

# ── Confidence thresholds ────────────────────────────────────────
# Crops with confidence BELOW this are 'unsure' -> good candidates for review
CONF_THRESHOLD_LOW  = 0.70   # below this = unsure, send for review
CONF_THRESHOLD_HIGH = 0.95   # above this = confident, skip review (unless forced)

# ── Which classes to include ─────────────────────────────────────
INCLUDE_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'background']

# ── Include background crops? ────────────────────────────────────
# Background crops from background/ folder are useful as hard negatives
INCLUDE_BACKGROUND = True

# ── Force include all (ignore confidence filter) ─────────────────
FORCE_ALL = False

# ── Max crops per class (to keep review manageable) ──────────────
MAX_PER_CLASS = 200   # None = no limit

print('Filter config ready.')


## Cell 3 — Load and inspect results

Reads all `results.csv` files from the selected run.
Shows confidence distribution per class so you can tune `CONF_THRESHOLD_LOW`
before deciding what to send for review.

In [ ]:
import csv
import numpy as np
from collections import defaultdict

all_rows = []
for csv_path in sorted(INFER_RESULTS.rglob('results.csv')):
    with open(csv_path, newline='') as f:
        for row in csv.DictReader(f):
            if row.get('pollinator_detected') not in ('yes', 'no'): continue
            row['_csv_path'] = str(csv_path)
            row['_camera']   = csv_path.parent.name
            all_rows.append(row)

print(f'Total rows loaded: {len(all_rows)}')

# ── Confidence distribution ──────────────────────────────────────
detected = [r for r in all_rows if r.get('pollinator_detected') == 'yes']
print(f'\nDetected (insect): {len(detected)}')

by_class = defaultdict(list)
for r in detected:
    pt = r.get('pollinator_type', '') or 'background'
    by_class[pt].append(r)

print('\nClass breakdown + confidence stats:')
print(f'  {"Class":15}  {"Count":>6}  {"AvgConf":>8}  {"<threshold":>10}')
for cls, rows in sorted(by_class.items()):
    confs = []
    for r in rows:
        try: confs.append(float(r.get('group_confidence') or r.get('binary_confidence') or 0))
        except: pass
    if confs:
        avg  = np.mean(confs)
        low  = sum(1 for c in confs if c < CONF_THRESHOLD_LOW)
        print(f'  {cls:15}  {len(rows):>6}  {avg:>8.3f}  {low:>10}')

# Background crops
bg_crops = list(INFER_RESULTS.rglob('background/*.jpg'))
print(f'\nBackground crops: {len(bg_crops)}')


## Cell 4 — Preview

Shows exactly how many crops will be sent per class **before** copying anything.
Review this table and adjust the filter config in Cell 2 if needed.
**Nothing is copied yet** — this is just a preview.

In [ ]:
to_review = defaultdict(list)  # class -> list of (crop_path, row)

# ── Insect crops (from results.csv) ─────────────────────────────
for r in detected:
    pt   = r.get('pollinator_type', '') or 'other'
    if pt not in INCLUDE_CLASSES: continue
    fname = r.get('crop_filename', '')
    if not fname: continue
    camera = r['_camera']
    crop_p = INFER_RESULTS / camera / 'crops' / fname
    if not crop_p.exists(): continue

    try: conf = float(r.get('group_confidence') or r.get('binary_confidence') or 0)
    except: conf = 0.0

    if FORCE_ALL or conf < CONF_THRESHOLD_LOW:
        to_review[pt].append((crop_p, conf, r))

# ── Background crops ─────────────────────────────────────────────
if INCLUDE_BACKGROUND:
    for bg_p in bg_crops:
        to_review['background'].append((bg_p, 0.0, {}))

# ── Apply max per class ──────────────────────────────────────────
# Sort by confidence ascending (most uncertain first)
print('Preview — crops to send for review:')
print(f'  {"Class":15}  {"Available":>10}  {"Will send":>10}')
total_send = 0
for cls in sorted(to_review):
    items = sorted(to_review[cls], key=lambda x: x[1])  # lowest conf first
    if MAX_PER_CLASS: items = items[:MAX_PER_CLASS]
    to_review[cls] = items
    total_send += len(items)
    print(f'  {cls:15}  {len(to_review[cls]):>10}  {len(items):>10}')
print(f'\nTotal: {total_send} crops')


## Cell 5 — Copy to review folder

Shows the final plan. **Still nothing copied yet.**
Run **Cell 6** (the Execute cell) to actually copy the files.

In [ ]:
import shutil

# Confirm before copying
print(f'About to copy {total_send} crops to:')
print(f'  {REVIEW_DIR}')
print()
print('Structure will be:')
for cls in sorted(to_review):
    print(f'  retrain_review/{cls}/  ({len(to_review[cls])} crops)')
print()
print('Run Cell 6 to confirm and execute.')


In [ ]:
# ── EXECUTE — run this cell to actually copy ─────────────────────
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

copied = defaultdict(int)
skipped = 0

for cls, items in to_review.items():
    dest_dir = REVIEW_DIR / cls
    dest_dir.mkdir(exist_ok=True)
    for crop_p, conf, row in items:
        dest = dest_dir / crop_p.name
        if dest.exists():  # don't overwrite
            skipped += 1; continue
        shutil.copy2(str(crop_p), str(dest))
        copied[cls] += 1

print('Done. Copied:')
for cls, n in sorted(copied.items()):
    print(f'  {cls:15}: {n}')
if skipped: print(f'Skipped (already exist): {skipped}')
print(f'\nReview folder: {REVIEW_DIR}')
print(f'\nNext: run crop_labeler_v2.py on {REVIEW_DIR}')
print(f'  python3 crop_labeler_v2.py --results {REVIEW_DIR}')
print(f'\nAfter labeling, manually move confirmed crops to:')
print(f'  {LABELED_DIR}')


## Cell 7 — Check labeled_crops (optional)

Shows current counts in `labeled_crops/` per class.
Useful to see how much data you have before deciding whether to retrain.

In [ ]:
print('Current labeled_crops counts:')
total = 0
for cls_dir in sorted(LABELED_DIR.iterdir()):
    if not cls_dir.is_dir(): continue
    n = len(list(cls_dir.glob('*.jpg'))) + len(list(cls_dir.glob('*.jpeg')))
    print(f'  {cls_dir.name:15}: {n:>5}')
    total += n
print(f'  {"TOTAL":15}: {total:>5}')
